In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from matplotlib.animation import FuncAnimation
import matplotlib.animation as animation
from matplotlib import cm
import numpy as np

print("numpy version:", np.__version__)
import sys
import argparse
import os
import glob
import tqdm

import torch
import torch.nn as nn
import tqdm

from utils_weights import (
    get_vanilla_rnn_weights_and_eigenvalues,
    plot_initial_final_eigenvalues,
    animate_eigenvalue_evolution,
    plot_top_k_eigenvalues,
    get_taus_from_J,
    get_taus_from_eigs,
    plot_initial_final_timescales,
    animate_timescale_evolution,                        
)
import utils_weights as wu

print(os.getcwd())

In [ ]:
# Interactive mode options (overwritten below by command line args)

# BASE_PATH = "../../../../../srv/marl/satsingh/marl_fish/"
# BASE_PATH = "results/rmappo-MultiAgentFishEnv-10K/20250526_171653/"  # Orthogonal Init
# BASE_PATH = "results/rmappo-MultiAgentFishEnv-10K/20250526_173336/"  # Xavier Uniform

# Weight decay experiments
BASE_PATH="results/rmappo-MultiAgentFishEnv-2MAttn/20250708_092959/"

MODELS_SUBFOLDER = "models/"
model_dir = BASE_PATH + MODELS_SUBFOLDER



In [ ]:
# Check if we're in interactive mode or batch mode
batchmode = False
if "ipykernel_launcher" in sys.argv[0]:
    print("Interactive mode")
else:
    batchmode = True
    print("Batch/CLI mode")

if batchmode:
    parser = argparse.ArgumentParser()
    parser.add_argument("--run_dir", 
                        type=str, 
                        help="Path to the run directory containing models/ sub-directory")
    args = parser.parse_args()
    run_dir = args.run_dir
    if run_dir is None:
        raise ValueError(parser.format_help())
    if not os.path.exists(run_dir):
        raise ValueError(f"Run directory {run_dir} does not exist.")
    model_dir = os.path.join(run_dir, MODELS_SUBFOLDER)
    print(f"run_dir: {run_dir}")

In [ ]:

print(f"Loading models from: {model_dir}")
all_actor_files = glob.glob(os.path.join(model_dir, "actor_*.pt"))
initial_file = [f for f in all_actor_files if "initial" in f][0]
episode_files = sorted(
    [f for f in all_actor_files if "initial" not in f],
    key=lambda x: int(x.split("_")[-1].split(".")[0]),
)
actor_files = [initial_file] + episode_files
# print(f"Found {len(actor_files)} actor files: {actor_files}")
print(f"Found {len(actor_files)} actor files")
episodes = ["initial"] + [int(f.split("_")[-1].split(".")[0]) for f in episode_files]
# print(episodes)
print("Number of episodes:", len(episodes))

In [ ]:
# Vanilla RNN weights extraction and saving

CACHE_SUBDIR = "weights_eigs_cached"
save_dir = os.path.join(model_dir, CACHE_SUBDIR)
os.makedirs(save_dir, exist_ok=True)
rnn_weights, eigenvalues = get_vanilla_rnn_weights_and_eigenvalues(actor_files, episodes, save_dir)


In [ ]:

out_filename = os.path.join(model_dir, "eigenvalues_initial_final.png")
plot_initial_final_eigenvalues(eigenvalues, episodes, out_filename, show=True)


In [ ]:
# ---- Animate eigenvalue evolution ----



ani_path = os.path.join(model_dir, "rnn_eigenvalues_animation.mp4")
animate_eigenvalue_evolution(eigenvalues, episodes, ani_path)


In [ ]:
# ---- Plot top-k eigenvalue magnitudes over time ----


out_filename = os.path.join(model_dir, "rnn_topk_eigenvalue_magnitudes.png")
plot_top_k_eigenvalues(eigenvalues, episodes, top_k=5, out_filename=out_filename)

In [ ]:
# Initial vs. final Timescales from eigenvalues assuming linear RNN dynamics
plot_initial_final_timescales(eigenvalues, model_dir)

In [ ]:

# Animate timescales evolution
ani_path = os.path.join(model_dir, "rnn_timescales_animation.mp4")
animate_timescale_evolution(eigenvalues, ani_path, num_modes=len(eigenvalues[0]))